# Netflix Data Analysis & Business Insights

**Internship:** Auspify Technologies - Data Analysis Using Python

This notebook walks through the six task themes: data cleaning, content type, country representation, release-year trends, ratings/genres, and business insights. Run it from the project root folder. The complete repeatable pipeline is also available in `src/analysis.py`.

## 0. Import libraries and helper functions

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from src.analysis import clean_dataset, split_multi_value

ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'Dataset.csv'
OUTPUT_DIR = ROOT / 'outputs'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

if not DATA_PATH.exists():
    raise FileNotFoundError('Open this notebook from the project root so data/Dataset.csv can be found.')

## 1. Load the raw dataset and inspect its quality
First check the number of records, column names, data types, missing values, duplicate rows, and common placeholder strings.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
display(raw.head())
display(raw.dtypes.rename('data_type').to_frame())
display(raw.isna().sum().rename('missing_values').to_frame())
print('Exact duplicate rows:', raw.duplicated().sum())
for column in ['director', 'country', 'title']:
    if column in raw.columns:
        print(f"{column} placeholder values:", raw[column].astype('string').str.strip().str.casefold().isin(['not given', 'unknown']).sum())

## Task 1 - Clean and prepare the dataset
The cleaning function standardizes text, replaces common placeholder values, normalizes Movie/TV Show labels, parses dates, creates numeric duration fields, checks duplicate IDs, and removes records that do not have a usable title.

In [ ]:
cleaned, quality = clean_dataset(raw)
cleaned.to_csv(ROOT / 'data' / 'cleaned_netflix.csv', index=False)
display(pd.DataFrame([quality]).T.rename(columns={0: 'value'}))
display(cleaned.head())
print('Cleaned dataset shape:', cleaned.shape)

## Task 2 - Content type analysis
Compare the number and percentage of Movies and TV Shows in the cleaned catalog.

In [ ]:
content_type = cleaned['type'].value_counts().rename_axis('content_type').reset_index(name='title_count')
content_type['percentage'] = (content_type['title_count'] / content_type['title_count'].sum() * 100).round(2)
display(content_type)
ax = sns.barplot(data=content_type, x='content_type', y='title_count', color='steelblue')
ax.set(title='Netflix Catalog: Movies vs TV Shows', xlabel='Content type', ylabel='Number of titles')
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f')
plt.tight_layout(); plt.show()

## Task 3 - Country-wise content analysis
Country fields can contain multiple countries. Explode the field so every listed country receives a title-country assignment. A single title may count for multiple countries.

In [ ]:
country_values = split_multi_value(cleaned['country'])
country_summary = country_values[country_values.ne('Unknown')].value_counts().head(15).rename_axis('country').reset_index(name='title_count')
display(country_summary.head(10))
ax = sns.barplot(data=country_summary, y='country', x='title_count', color='steelblue')
ax.set(title='Top 15 Countries by Netflix Catalog Titles', xlabel='Title-country assignments', ylabel='Country')
plt.tight_layout(); plt.show()

## Task 4 - Release-year trend
This represents release years of titles in this dataset; it does not represent all worldwide productions.

In [ ]:
year_summary = cleaned.groupby('release_year').size().rename('title_count').reset_index()
display(year_summary.sort_values('title_count', ascending=False).head(10))
recent = year_summary[year_summary['release_year'] >= 2000]
plt.figure(figsize=(11, 5))
sns.lineplot(data=recent, x='release_year', y='title_count', marker='o')
plt.title('Netflix Catalog Titles by Release Year (2000 onward)')
plt.xlabel('Release year'); plt.ylabel('Number of titles in dataset')
plt.tight_layout(); plt.show()

## Task 5 - Ratings and genre analysis
Analyze content rating categories, compare Movie and TV Show ratings, and find the most common genre/category assignments.

In [ ]:
rating_summary = cleaned['rating'].value_counts().rename_axis('rating').reset_index(name='title_count')
display(rating_summary.head(12))
rating_by_type = cleaned.groupby(['rating', 'type']).size().rename('title_count').reset_index()
top_ratings = rating_summary.head(12)['rating']
rating_plot = rating_by_type[rating_by_type['rating'].isin(top_ratings)]
sns.barplot(data=rating_plot, x='rating', y='title_count', hue='type')
plt.title('Content Ratings by Type'); plt.xlabel('Rating'); plt.ylabel('Number of titles')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()

genre_values = split_multi_value(cleaned['listed_in'])
genre_summary = genre_values[genre_values.ne('Unknown')].value_counts().head(15).rename_axis('genre').reset_index(name='title_count')
display(genre_summary.head(10))
sns.barplot(data=genre_summary, y='genre', x='title_count', color='steelblue')
plt.title('Top 15 Genres / Categories'); plt.xlabel('Category assignments'); plt.ylabel('Genre / category')
plt.tight_layout(); plt.show()

## Task 6 - Business insights and report
Use the outputs above to write a business-focused summary. Avoid claiming the charts prove causes or audience demographics; they describe the catalog represented in this dataset.

In [ ]:
print('CONTENT MIX')
display(content_type)
print('TOP COUNTRY ASSIGNMENTS')
display(country_summary.head(5))
print('TOP RATING')
display(rating_summary.head(1))
print('TOP GENRE / CATEGORY')
display(genre_summary.head(5))

# Save a few analysis tables for the submission folder.
content_type.to_csv(OUTPUT_DIR / 'content_type_summary_from_notebook.csv', index=False)
country_summary.to_csv(OUTPUT_DIR / 'country_summary_from_notebook.csv', index=False)
year_summary.to_csv(OUTPUT_DIR / 'release_year_summary_from_notebook.csv', index=False)
rating_summary.to_csv(OUTPUT_DIR / 'rating_summary_from_notebook.csv', index=False)
genre_summary.to_csv(OUTPUT_DIR / 'genre_summary_from_notebook.csv', index=False)
print('Summary tables saved in outputs/.')

## Submission checklist
- [ ] Run all notebook cells without errors.
- [ ] Confirm `data/cleaned_netflix.csv` was created.
- [ ] Review charts and tables.
- [ ] Run the Streamlit dashboard using `streamlit run dashboard.py`.
- [ ] Capture screenshots and include the README, source code, report and selected outputs in your submission.

**Finding from this dataset:** Movies are the larger content type (about 69.7%); TV Shows are about 30.3%. The United States is the most represented country by title-country assignments, and International Movies is the most frequent category assignment.